# Export model simulations and TE

**Notebook contents:**
* Exporting of D'Orsogna simulations and TE across all aggregation schemes (all-pairs, fixed-radius, nearest-neighbors)
* Exporting of baseline models simulations and TE across all-pairs aggregation scheme

Notes:
* to export simulations, change `animate` paramater values to `True`
* to export TE csvs, run `compute_modelTE`

In [ ]:
import os
from main import DorsognaTE, RandomWalkTE, CorrRandomWalkTE, DorsognaNoisyTE

# D'Orsogna phenotypes

In [ ]:
phenotype_dict = {
    'singlemill':[0.5,0.1],
    'doublemill':[0.9,0.5],
    'doublering':[0.5,0.5],
    'collswarm':[0.5,0.9],
    'escapesymm':[2,0.9],
    'escapeunsymm':[2,2],
    'escapecoll':[2,3]
    }

phenotype_fr_lengths = {
    'singlemill':[0.0625,0.125],
    'doublemill':[0.125,0.25],
    'doublering':[0.625,0.125],
    'collswarm':[0.03,0.05],
    'escapesymm':[20,40],
    'escapeunsymm':[20,40],
    'escapecoll':[20,40]
}

In [ ]:
output_dir = 'csvs_actual_te_values'

for phenotype_name, values in phenotype_dict.items():
    C, l = values
    os.makedirs(output_dir, exist_ok=True)
    insights_path = os.path.join(output_dir, phenotype_name)
    model = DorsognaTE(outdir=insights_path)
    model.develop_model(
            C=C,
            l=l,
            phenotype_name=phenotype_name,
            particle_count=100,
            t_max=40,
            vel_scale=0.1,
            fps=20,
            dt=0.1,
            animate=False
        )

    # Calculate all-pairs scheme
    for TE_ver in ['linvel','angvel']:
        model.compute_modelTE(TE_ver = TE_ver,
                      TE_embedding = 15,
                      save_graph = False)

    # Calculate fixed-radius scheme
    for TE_ver in ['linvel','angvel']:
        for fr_length in phenotype_fr_lengths[phenotype_name]:
            model.compute_modelTE(TE_ver = TE_ver,
                        TE_embedding = 15,
                        neighbor_radius = fr_length,
                        save_graph = False)

    # Calculate nearest-neighbors scheme
    for TE_ver in ['linvel','angvel']:
        for nn_count in [10,30]:
            model.compute_modelTE(TE_ver = TE_ver,
                        TE_embedding = 15,
                        nearest_neighbors = nn_count,
                        save_graph = False)
    
    print("\n")

# Null models

In [ ]:
folder = 'csvs_baseline'
os.makedirs(folder, exist_ok=True)

## Random walk

In [ ]:
# Random walk
insights_path = os.path.join(folder, 'randomwalk')
rw_model = RandomWalkTE(outdir=insights_path)

rw_model.develop_model(
        particle_count=100,
        sigma=1.0,
        t_max=40,
        vel_scale=1,
        seed = 1,
        fps=10,
        dt=0.1,
        animate=False,
        trail_length = 10
    )

for TE_ver in ['linvel','angvel']:
    rw_model.compute_modelTE(TE_ver = TE_ver,
                    TE_embedding = 15,
                    save_graph = False)

## Correlated random walk

In [ ]:
# Correlated random walk
insights_path = os.path.join(folder, 'corrrandomwalk')
os.makedirs(insights_path, exist_ok=True)
corr_rw_model = CorrRandomWalkTE(outdir=insights_path)

for kappa in [1,4,8]:
    corr_rw_model.develop_model(
        particle_count= 100,
        kappa = kappa,
        t_max=40,
        seed = 1,
        vel_scale=1,
        fps=10,
        dt=0.1,
        animate=False,
        trail_length=10
    )

    for TE_ver in ['linvel','angvel']:
        corr_rw_model.compute_modelTE(TE_ver = TE_ver,
                        TE_embedding = 15,
                        save_graph = False)

## D'Orsogna noisy

In [ ]:
# D'orsogna noisy
insights_path = os.path.join(folder, 'dorsognanoisy')
os.makedirs(insights_path, exist_ok=True)
do_noise_model = DorsognaNoisyTE(outdir=insights_path)

for diff_coef in [0.1,2]:
    do_noise_model.develop_model(
            particle_count=100,
            phenotype_name= "dorsognanoisy",
            t_max=40,
            vel_scale=0.1,
            seed = 1,
            diff_coef = diff_coef, 
            fps=10,
            dt=0.1,
            animate=False,
            trail_length=10
        )
    
    for TE_ver in ['linvel','angvel']:
        do_noise_model.compute_modelTE(TE_ver = TE_ver,
                        TE_embedding = 15,
                        save_graph = False)